# NB4 — Medallion Pipeline (Bronze → Silver → Gold), lightweight

**Use case:** LLM observability — exact schema from slide §8 (Lakehouse cho AI/ML) medallion frame.
Maps to deliverable bullet 4 (the Milestone-1 Lakehouse artifact).

Pre-req: ran `make data` — but if you jumped straight here, the cell below
generates the Bronze sample for you rather than failing on a missing path.

In [1]:
import _setup  # noqa: F401  -- adds scripts/ to sys.path
from pathlib import Path

import polars as pl
import duckdb
from deltalake import DeltaTable, write_deltalake
from lakehouse import path, reset

BRONZE = path("bronze", "llm_calls_raw")
SILVER = path("silver", "llm_calls")
GOLD   = path("gold",   "llm_daily_metrics")

# Self-healing pre-req (same pattern as NB7/NB8). Without this, skipping
# `make data` surfaces as a raw `Os { code: 2, kind: NotFound }` from the Rust
# layer — technically correct, useless to a student.
if not Path(BRONZE).exists():
    print("Bronze not found — running scripts/generate_data_lite.py first ...")
    import generate_data_lite

    generate_data_lite.main()

## Bronze — verify raw is loaded

In [2]:
bronze_n = DeltaTable(BRONZE).to_pyarrow_table().num_rows
print(f"Bronze rows: {bronze_n:,}")
print(pl.from_arrow(DeltaTable(BRONZE).to_pyarrow_table().slice(0, 2)))

Bronze rows: 200,000
shape: (2, 3)
┌─────────────────────────────────┬─────────────────────────┬─────────────────────────────────┐
│ request_id                      ┆ ts                      ┆ raw_json                        │
│ ---                             ┆ ---                     ┆ ---                             │
│ str                             ┆ datetime[μs, UTC]       ┆ str                             │
╞═════════════════════════════════╪═════════════════════════╪═════════════════════════════════╡
│ 19ab58fa-5db7-4375-9117-a771eb… ┆ 2026-04-01 00:00:00 UTC ┆ {"model": "claude-sonnet-4-6",… │
│ 2591a144-d63e-4d08-95cb-744fd5… ┆ 2026-04-01 00:00:03 UTC ┆ {"model": "claude-haiku-4-5", … │
└─────────────────────────────────┴─────────────────────────┴─────────────────────────────────┘


## Silver — parse, validate, dedup

Rules: drop malformed JSON, dedupe by `request_id`, project typed columns.

In [3]:
reset(SILVER)

# DuckDB does the JSON parse + dedup in one query — Polars also works,
# DuckDB just has nicer JSON syntax for this case.
# DuckDB reads Delta through Arrow, not through `delta_scan()`. The latter
# autoloads an extension over the network; Arrow registration is offline and
# zero-copy, so the lab works on a locked-down machine.
con = duckdb.connect()
con.register("bronze", DeltaTable(BRONZE).to_pyarrow_table())

silver_arrow = con.sql(f"""
    WITH parsed AS (
      SELECT
        request_id,
        ts,
        CAST(ts AS DATE)                            AS date,
        json_extract_string(raw_json, '$.model')          AS model,
        json_extract_string(raw_json, '$.user_id')        AS user_id,
        CAST(json_extract(raw_json, '$.usage.input')  AS INTEGER) AS prompt_tokens,
        CAST(json_extract(raw_json, '$.usage.output') AS INTEGER) AS completion_tokens,
        CAST(json_extract(raw_json, '$.latency_ms')   AS INTEGER) AS latency_ms,
        json_extract_string(raw_json, '$.status')         AS status,
        ROW_NUMBER() OVER (PARTITION BY request_id ORDER BY ts) AS rn
      FROM bronze
    )
    SELECT request_id, ts, date, model, user_id,
           prompt_tokens, completion_tokens, latency_ms, status
    FROM parsed
    WHERE rn = 1 AND model IS NOT NULL
""").arrow()

write_deltalake(SILVER, silver_arrow, mode="overwrite", partition_by=["date"])

silver_n = DeltaTable(SILVER).to_pyarrow_table().num_rows
print(f"Silver rows: {silver_n:,}  (Bronze {bronze_n:,} → dedup dropped {bronze_n - silver_n:,})")
assert silver_n < bronze_n, (
    "Silver has the same row count as Bronze — dedup did not run. "
    "Did you regenerate Bronze with the latest generator (which injects retries)?"
)

Silver rows: 190,052  (Bronze 200,000 → dedup dropped 9,948)


## Gold — aggregate to (date, model) metrics

In [4]:
reset(GOLD)

# Illustrative cost model — NOT canonical pricing.
# (input USD / 1M tokens, output USD / 1M tokens)
COST_TABLE = """
  VALUES
    ('claude-haiku-4-5',  0.80,  4.00),
    ('claude-sonnet-4-6', 3.00, 15.00),
    ('claude-opus-4-7', 15.00, 75.00)
"""

con.register("silver", DeltaTable(SILVER).to_pyarrow_table())
gold_arrow = con.sql(f"""
    WITH cost(model, c_in, c_out) AS ({COST_TABLE})
    SELECT
      s.date,
      s.model,
      QUANTILE_CONT(s.latency_ms, 0.50) AS p50_latency_ms,
      QUANTILE_CONT(s.latency_ms, 0.95) AS p95_latency_ms,
      SUM(s.prompt_tokens)              AS total_prompt_tokens,
      SUM(s.completion_tokens)          AS total_completion_tokens,
      AVG(CASE WHEN s.status <> 'ok' THEN 1.0 ELSE 0.0 END) AS error_rate,
      (SUM(s.prompt_tokens)     * c.c_in  / 1e6) +
      (SUM(s.completion_tokens) * c.c_out / 1e6) AS cost_usd
    FROM silver s
    JOIN cost c USING (model)
    GROUP BY s.date, s.model, c.c_in, c.c_out
    ORDER BY s.date, s.model
""").arrow()

write_deltalake(GOLD, gold_arrow, mode="overwrite", partition_by=["date"])

# Z-order for fast filter-by-model dashboards
DeltaTable(GOLD).optimize.z_order(["model"])

{'numFilesAdded': 8,
 'numFilesRemoved': 8,
 'filesAdded': '{"avg":2768.75,"max":2770,"min":2766,"totalFiles":8,"totalSize":22150}',
 'filesRemoved': '{"avg":2687.875,"max":2689,"min":2687,"totalFiles":8,"totalSize":21503}',
 'partitionsOptimized': 8,
 'numBatches': 8,
 'totalConsideredFiles': 8,
 'totalFilesSkipped': 0,
 'preserveInsertionOrder': False,
 'plannerStrategy': 'zOrder',
 'preservedStableOrder': False,
 'maxBinSpanFiles': 1}

## Verify Gold

In [5]:
gold_df = pl.from_arrow(DeltaTable(GOLD).to_pyarrow_table())
print(gold_df)

# Slide-5 deliverable: "Gold p50/p95/cost qua ≥ 7 ngày". Make that explicit.
n_dates = gold_df.select("date").n_unique()
n_models = gold_df.select("model").n_unique()
print(
    f"\n──── Gold deliverable metrics ────\n"
    f"  Distinct dates:   {n_dates:>3}   (target ≥ 7)\n"
    f"  Distinct models:  {n_models:>3}\n"
    f"  Total Gold rows:  {gold_df.height:>3}   (= dates × models)"
)
assert n_dates >= 7, (
    f"Gold has only {n_dates} dates — slide deliverable requires ≥ 7. "
    "Re-run `make data` (the generator spreads across 7 UTC days)."
)

shape: (24, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ date       ┆ model      ┆ p50_latenc ┆ p95_laten ┆ total_pro ┆ total_com ┆ error_rat ┆ cost_usd  │
│ ---        ┆ ---        ┆ y_ms       ┆ cy_ms     ┆ mpt_token ┆ pletion_t ┆ e         ┆ ---       │
│ date       ┆ str        ┆ ---        ┆ ---       ┆ s         ┆ okens     ┆ ---       ┆ f64       │
│            ┆            ┆ f64        ┆ f64       ┆ ---       ┆ ---       ┆ f64       ┆           │
│            ┆            ┆            ┆           ┆ decimal[3 ┆ decimal[3 ┆           ┆           │
│            ┆            ┆            ┆           ┆ 8,0]      ┆ 8,0]      ┆           ┆           │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 2026-04-01 ┆ claude-hai ┆ 567.0      ┆ 1121.0    ┆ 11888206  ┆ 5916990   ┆ 0.04936   ┆ 33.178525 │
│            ┆ ku-4-5     ┆            ┆           ┆           ┆           ┆

## ✅ Deliverable check
- [ ] All three tables exist under `_lakehouse/{bronze,silver,gold}/`
- [ ] Silver has fewer rows than Bronze (dedup worked)
- [ ] Gold spans ≥ 7 dates × 3 models (slide §8 medallion contract)
- [ ] Cost & error_rate columns populated and non-zero

In [6]:
# --- Evidence cell (added by submitter): validate every Gold requirement the notebook does not assert ---
import polars as pl
from deltalake import DeltaTable
from lakehouse import path as _lp

_gold = pl.from_arrow(DeltaTable(_lp("gold", "llm_daily_metrics")).to_pyarrow_table()).sort("date", "model")
print(_gold.select("date", "model", "p50_latency_ms", "p95_latency_ms", "error_rate", "cost_usd"))
_checks = {
    ">= 7 distinct dates": _gold["date"].n_unique() >= 7,
    "3 models": _gold["model"].n_unique() == 3,
    "every (date, model) pair present (dates x models rows)": _gold.height == _gold["date"].n_unique() * 3,
    "p50 <= p95 on every row": bool((_gold["p50_latency_ms"] <= _gold["p95_latency_ms"]).all()),
    "cost_usd > 0 on every row": bool((_gold["cost_usd"] > 0).all()),
    "error_rate in [0, 1] on every row": bool(_gold["error_rate"].is_between(0, 1).all()),
}
for k, v in _checks.items():
    print(f"  [{'PASS' if v else 'FAIL'}] {k}")
assert all(_checks.values())


shape: (24, 6)
┌────────────┬───────────────────┬────────────────┬────────────────┬────────────┬────────────┐
│ date       ┆ model             ┆ p50_latency_ms ┆ p95_latency_ms ┆ error_rate ┆ cost_usd   │
│ ---        ┆ ---               ┆ ---            ┆ ---            ┆ ---        ┆ ---        │
│ date       ┆ str               ┆ f64            ┆ f64            ┆ f64        ┆ f64        │
╞════════════╪═══════════════════╪════════════════╪════════════════╪════════════╪════════════╡
│ 2026-04-01 ┆ claude-haiku-4-5  ┆ 567.0          ┆ 1121.0         ┆ 0.04936    ┆ 33.178525  │
│ 2026-04-01 ┆ claude-opus-4-7   ┆ 3045.0         ┆ 6035.65        ┆ 0.049735   ┆ 200.248575 │
│ 2026-04-01 ┆ claude-sonnet-4-6 ┆ 1381.0         ┆ 2752.0         ┆ 0.050408   ┆ 244.145982 │
│ 2026-04-02 ┆ claude-haiku-4-5  ┆ 565.0          ┆ 1140.0         ┆ 0.049327   ┆ 46.305166  │
│ 2026-04-02 ┆ claude-opus-4-7   ┆ 3004.0         ┆ 5972.0         ┆ 0.052803   ┆ 291.28575  │
│ …          ┆ …                 ┆ 

## Phân tích kết quả NB4

* **Ba tầng trên đĩa**: Bronze `_lakehouse/bronze/llm_calls_raw` (200,000 dòng JSON thô), Silver `_lakehouse/silver/llm_calls`, Gold `_lakehouse/gold/llm_daily_metrics`.
* **Dedup**: Silver **190,052 < Bronze 200,000** - đúng 9,948 bản trùng `request_id` được cài sẵn trong dữ liệu mẫu đã bị loại; Bronze giữ nguyên dữ liệu thô để có thể tái xử lý.
* **Gold**: 8 ngày x 3 model = **24 dòng** (>= 7 ngày x 3 model). Cell bằng chứng cuối notebook kiểm tra thêm các điều kiện mà notebook gốc không assert: mọi dòng có `p50 <= p95`, `cost_usd > 0`, `error_rate` trong [0, 1]. Các model đắt/chậm hơn cho p50/p95/cost cao hơn rõ rệt (opus p50 ~3.0 s so với haiku ~0.57 s) - đúng thứ tự mong đợi.
* Lưu ý: ngày 2026-04-08 có token/chi phí thấp hơn hẳn các ngày khác vì dữ liệu mẫu chỉ trải khoảng 7 ngày UTC nên ngày thứ 8 chỉ có phần đuôi; giá token là giá minh họa của lab.
